In [1]:
## READS MNQ-TICK from HGT-Export SC chartbook

In [2]:
import pandas as pd
import numpy as np
import datetime as dt 
from pathlib import Path
import time

In [3]:
# Configuration: Style preferences
#plt.style.use('ggplot') # Good default for readability
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [4]:
import os
os.getcwd()

'/home/vm/pt/hgt-rl/mnq-tick'

In [5]:
%%time
symbol = 'mnq'
SEC = 2

inFile = f'/mnt/d/SierraChart/data/EXPORT/MNQ-CME-OHLC.csv'
outFile= f'data/mnq-ohlc-raw-6sec.pqt'

print(inFile, outFile,)

/mnt/d/SierraChart/data/EXPORT/MNQ-CME-OHLC.csv data/mnq-ohlc-raw-6sec.pqt
CPU times: user 92 μs, sys: 7 μs, total: 99 μs
Wall time: 95.1 μs


In [12]:
df = pd.read_csv(inFile)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 5512771 entries, 0 to 5512770
Data columns (total 10 columns):
 #   Column          Dtype  
---  ------          -----  
 0   Date            str    
 1   Time            str    
 2   Open            float64
 3   High            float64
 4   Low             float64
 5   Last            float64
 6   Volume          int64  
 7   NumberOfTrades  int64  
 8   BidVolume       int64  
 9   AskVolume       int64  
dtypes: float64(4), int64(4), str(2)
memory usage: 509.6 MB
None
       Date      Time     Open      High       Low      Last  Volume  NumberOfTrades  BidVolume  AskVolume
0  2022/1/3  08:00:00  16431.0  16431.00  16429.00  16429.25      83              83         48         35
1  2022/1/3  08:00:06  16429.5  16430.00  16428.75  16429.50      40              40         15         25
2  2022/1/3  08:00:12  16429.5  16432.25  16429.25  16431.25      65              63         18         47
3  2022/1/3  08:00:18  16431.5  16432.25  16430.75  16432

In [13]:
# expand stupid SC date like 2026-7-8 to 2026-07-08
parts = df['Date'].astype(str).str.split('/', expand=True)

year  = parts[0]
month = parts[1].str.zfill(2)
day   = parts[2].str.zfill(2)

df['Date_norm'] = year + '-' + month + '-' + day

# join date and time and convert to wall clock datetime64
df['timestamp'] = pd.to_datetime(
    df['Date_norm'] + ' ' + df['Time'].astype(str),
    utc=False,            # keep as wall clock, no timezone
    errors='raise'        # or 'coerce' if you want bad rows as NaT
)

# remove temp columns
df = df.drop(columns=['Date', 'Date_norm', 'Time'])

# make timestamp first column
col = df.pop('timestamp')
df.insert(0, 'timestamp', col)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 5512771 entries, 0 to 5512770
Data columns (total 9 columns):
 #   Column          Dtype         
---  ------          -----         
 0   timestamp       datetime64[us]
 1   Open            float64       
 2   High            float64       
 3   Low             float64       
 4   Last            float64       
 5   Volume          int64         
 6   NumberOfTrades  int64         
 7   BidVolume       int64         
 8   AskVolume       int64         
dtypes: datetime64[us](1), float64(4), int64(4)
memory usage: 378.5 MB
None
            timestamp     Open      High       Low      Last  Volume  NumberOfTrades  BidVolume  AskVolume
0 2022-01-03 08:00:00  16431.0  16431.00  16429.00  16429.25      83              83         48         35
1 2022-01-03 08:00:06  16429.5  16430.00  16428.75  16429.50      40              40         15         25
2 2022-01-03 08:00:12  16429.5  16432.25  16429.25  16431.25      65              63         18         47

In [15]:
df.drop(columns=['Volume','NumberOfTrades','BidVolume','AskVolume'], inplace=True) 

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 5512771 entries, 0 to 5512770
Data columns (total 5 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   Open       float64       
 2   High       float64       
 3   Low        float64       
 4   Last       float64       
dtypes: datetime64[us](1), float64(4)
memory usage: 210.3 MB
None
            timestamp     Open      High       Low      Last
0 2022-01-03 08:00:00  16431.0  16431.00  16429.00  16429.25
1 2022-01-03 08:00:06  16429.5  16430.00  16428.75  16429.50
2 2022-01-03 08:00:12  16429.5  16432.25  16429.25  16431.25
3 2022-01-03 08:00:18  16431.5  16432.25  16430.75  16432.25
4 2022-01-03 08:00:24  16432.0  16432.00  16431.25  16431.25


In [16]:
print(f'monotonic: {df["timestamp"].is_monotonic_increasing}')

df['date'] = df['timestamp'].dt.normalize()

day = pd.Timestamp("2025-11-28")
df = df[df["date"] != day]

df = df[df['timestamp'].dt.time >= dt.time(9, 30, 0)]

print(df.info())
print(df.head())

monotonic: True
<class 'pandas.DataFrame'>
Index: 4461888 entries, 898 to 5512770
Data columns (total 6 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   Open       float64       
 2   High       float64       
 3   Low        float64       
 4   Last       float64       
 5   date       datetime64[us]
dtypes: datetime64[us](2), float64(4)
memory usage: 238.3 MB
None
              timestamp      Open      High       Low      Last       date
898 2022-01-03 09:30:00  16382.75  16395.25  16378.50  16395.25 2022-01-03
899 2022-01-03 09:30:06  16395.25  16397.25  16390.50  16395.75 2022-01-03
900 2022-01-03 09:30:12  16395.50  16402.25  16389.50  16401.75 2022-01-03
901 2022-01-03 09:30:18  16401.75  16409.00  16399.25  16407.25 2022-01-03
902 2022-01-03 09:30:24  16407.50  16413.00  16404.25  16405.25 2022-01-03


In [17]:
df.to_parquet(outFile, index=False)